# Notebook 01 — Coleta Inicial de Dados

**Propósito:** documentar e executar a coleta inicial dos dados para os três municípios via `siconfi-collector`. Não gera artefatos para o TCC, mas serve como executável reproduzível da etapa 1.

**Fonte única:** RREO-Anexo 03 (Demonstrativo da Receita Corrente Líquida). Ele traz a série mensal por tributo (colunas `<MR>`) **e** a coluna `PREVISÃO ATUALIZADA <ano>` por tributo — usada como projeção da prefeitura no benchmark. O RREO-Anexo 01 não é coletado porque seu rótulo Padrão só expõe a linha agregada `Impostos`, sem discriminar IPTU/ISSQN.

Coleta-se o bimestre **P6** de cada ano (fecha o calendário, 12 meses, e dá a previsão atualizada de fim de exercício). Opcionalmente, coletar também **P1** dá a previsão atualizada mais próxima da Previsão Inicial da LOA — veja o comentário na célula de coleta.

**Saída:** CSVs em `siconfi-collector/data/rreo/RREO-Anexo_03/` e `data/collection_report.json`.

In [ ]:
import json
from pathlib import Path

from forecasting.config import load_config
from siconfi.entities import EntityRegistry
from siconfi.collector import collect_rreo

cfg = load_config()
print(f'TCC root     : {cfg.tcc_root}')
print(f'siconfi data : {cfg.siconfi_data_dir}')
print(f'Janela       : {cfg.sample_window.start} a {cfg.sample_window.end}')
print(f'Municipios   : {[m.name for m in cfg.municipalities.values()]}')

In [ ]:
# Resolver entidades a partir dos codigos IBGE no config.
registry = EntityRegistry.from_api()
codes = [m.cod_ibge for m in cfg.municipalities.values()]
entities = registry.by_codes(codes)
for e in entities:
    print(f'  {e.cod_ibge}  {e.name:<20} {e.uf}  {e.population:>10,}')

In [ ]:
# RREO-Anexo 03: serie mensal por tributo + previsao atualizada por tributo.
# Janela 2015-2025 (anos anteriores: codificacao PCASP heterogenea). Coletar
# alguns anos a mais e inofensivo - a analise filtra a janela do .tcc-pipeline.json.
start_year = int(cfg.sample_window.start[:4])
end_year = int(cfg.sample_window.end[:4])
result = collect_rreo(
    entities=entities,
    years=list(range(start_year, end_year + 1)),
    periods=[6],          # P6 fecha o ano (12 meses). Opcional: adicione 1 para
                          # a 'Previsao Atualizada' do 1o bimestre (~ Previsao Inicial);
                          # nesse caso re-rode transform-monthly e confira a invariante
                          # de 12 meses/ano (P1 e P6 se sobrepoem em jan/fev).
    annex='RREO-Anexo 03',
    output_dir=cfg.siconfi_data_dir,
    delay=0.5,
)
print(f'Anexo 03: {result.successful} successful / {result.failed} failed')

In [ ]:
# Validacao de cobertura.
report = {
    'anexo03_files': result.successful,
    'anexo03_failed': result.failed,
    'window': [cfg.sample_window.start, cfg.sample_window.end],
    # TODO: enumerar lacunas por (municipio, ano) varrendo os CSVs gravados.
}
report_path = cfg.siconfi_data_dir / 'collection_report.json'
report_path.write_text(json.dumps(report, indent=2, ensure_ascii=False))
print(f'Relatorio salvo em: {report_path}')

In [ ]:
# Transformar para os formatos analiticos tabulares.
import subprocess
subprocess.run(['siconfi', '--data-dir', str(cfg.siconfi_data_dir),
                'transform-monthly', '--annex', 'RREO-Anexo_03'], check=True)
subprocess.run(['siconfi', '--data-dir', str(cfg.siconfi_data_dir),
                'transform-prefeitura-forecast', '--annex', 'RREO-Anexo_03'], check=True)